# 🖼️ CIFAR-10 Image Classifier — SVM & k-NN with OpenCV

**Simple image classification using classical ML — no deep learning, no GPU needed.**

| Step | What happens |
|------|--------------|
| 1 | Install & import libraries |
| 2 | Download CIFAR-10 dataset |
| 3 | Preprocess images with OpenCV |
| 4 | Show image enhancement demo |
| 5 | Train SVM and k-NN models |
| 6 | Evaluate with accuracy + confusion matrix |
| 7 | Visualise predictions |

> ▶️ **Runtime → Run all** to execute every cell in one click.

## ⚙️ Step 1 — Install & Import Libraries

In [ ]:
# Install opencv (headless version works perfectly on Colab)
!pip install -q opencv-python-headless scikit-learn torchvision

In [ ]:
import numpy as np
import cv2
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import warnings
warnings.filterwarnings('ignore')

from torchvision import datasets
from sklearn.model_selection import train_test_split
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

# ── Constants ─────────────────────────────────────────────────────────────────
IMG_SIZE     = 32       # resize target (matches CIFAR-10 native size)
MAX_SAMPLES  = 10_000   # subsample for speed; change to 60000 for full dataset
RANDOM_STATE = 42

CIFAR10_CLASSES = [
    'airplane', 'automobile', 'bird', 'cat', 'deer',
    'dog', 'frog', 'horse', 'ship', 'truck'
]

print('✅ Libraries loaded successfully!')

## 📦 Step 2 — Download CIFAR-10 Dataset

In [ ]:
print('Downloading CIFAR-10 (cached after first run)…')

train_data = datasets.CIFAR10(root='./data', train=True,  download=True)
test_data  = datasets.CIFAR10(root='./data', train=False, download=True)

# PIL images → numpy arrays
X_train_raw = np.array([np.array(img) for img, _ in train_data])  # (50000,32,32,3)
y_train_raw = np.array([lbl           for _, lbl in train_data])
X_test_raw  = np.array([np.array(img) for img, _ in test_data])   # (10000,32,32,3)
y_test_raw  = np.array([lbl           for _, lbl in test_data])

# Merge, then re-split reproducibly
X_all = np.concatenate([X_train_raw, X_test_raw], axis=0)
y_all = np.concatenate([y_train_raw, y_test_raw], axis=0)

# Stratified subsample so all 10 classes stay balanced
_, X_all, _, y_all = train_test_split(
    X_all, y_all,
    test_size=MAX_SAMPLES,
    stratify=y_all,
    random_state=RANDOM_STATE
)

print(f'✅ Using {len(X_all):,} images  |  shape: {X_all.shape}')

# Quick peek at the first 8 raw images
fig, axes = plt.subplots(1, 8, figsize=(16, 2.5))
for ax, i in zip(axes, range(8)):
    ax.imshow(X_all[i])
    ax.set_title(CIFAR10_CLASSES[y_all[i]], fontsize=9)
    ax.axis('off')
fig.suptitle('Sample CIFAR-10 Images (raw RGB)', fontsize=12, y=1.05)
plt.tight_layout()
plt.show()

## 🔧 Step 3 — Preprocess Images with OpenCV

Pipeline: **RGB → Grayscale → Resize → Normalize → Flatten**

In [ ]:
def preprocess(images):
    """
    Convert a batch of raw RGB images into normalised, flattened feature vectors.

    Returns array of shape (N, IMG_SIZE * IMG_SIZE) with float32 values in [0, 1].
    """
    processed = []
    for img in images:
        # 1. Grayscale — reduces 3-channel to 1-channel (1024 features per image)
        gray = cv2.cvtColor(img, cv2.COLOR_RGB2GRAY)

        # 2. Resize — standardise all images to IMG_SIZE × IMG_SIZE
        resized = cv2.resize(gray, (IMG_SIZE, IMG_SIZE), interpolation=cv2.INTER_AREA)

        # 3. Normalize — pixel values to [0, 1] for stable ML training
        normalized = resized.astype(np.float32) / 255.0

        # 4. Flatten — 2-D image → 1-D feature vector for SVM / k-NN
        processed.append(normalized.flatten())

    return np.array(processed, dtype=np.float32)


print('Preprocessing images…')
X = preprocess(X_all)   # shape → (MAX_SAMPLES, 1024)

# 80 / 20 train-test split
X_train, X_test, y_train, y_test, idx_train, idx_test = train_test_split(
    X, y_all, np.arange(len(X_all)),
    test_size=0.2,
    random_state=RANDOM_STATE,
    stratify=y_all
)
X_test_raw_subset = X_all[idx_test]   # raw images aligned with X_test

print(f'✅ Preprocessing done!')
print(f'   Train : {X_train.shape[0]:,} samples  |  shape per sample: {X_train.shape[1]}')
print(f'   Test  : {X_test.shape[0]:,}  samples')

## ✨ Step 4 — Image Enhancement Demo

Showing **Original | Brightness+Contrast | Gaussian Blur** side by side.

In [ ]:
def enhance_image(img):
    """
    Apply three enhancement techniques to a single RGB image.
    Returns: (gray, enhanced, blurred) — each upscaled to 256×256 for visibility.
    """
    DISP = 256

    # Grayscale baseline
    gray = cv2.cvtColor(img, cv2.COLOR_RGB2GRAY)
    gray = cv2.resize(gray, (DISP, DISP), interpolation=cv2.INTER_NEAREST)

    # Brightness (+40) & Contrast (×1.5)
    enhanced = cv2.convertScaleAbs(gray, alpha=1.5, beta=40)

    # Gaussian Blur — 7×7 kernel
    blurred = cv2.GaussianBlur(gray, (7, 7), sigmaX=0)

    return gray, enhanced, blurred


# Run on a few sample images
sample_indices = [0, 10, 50, 100]
fig, axes = plt.subplots(len(sample_indices), 4,
                         figsize=(14, 3.5 * len(sample_indices)))

col_titles = ['Original (RGB)', 'Grayscale', 'Brightness + Contrast', 'Gaussian Blur']
for col, title in enumerate(col_titles):
    axes[0][col].set_title(title, fontsize=11, fontweight='bold', pad=8)

for row, idx in enumerate(sample_indices):
    raw_img = X_all[idx]
    gray, enhanced, blurred = enhance_image(raw_img)

    # Original RGB (upscaled for display)
    axes[row][0].imshow(cv2.resize(raw_img, (256, 256), interpolation=cv2.INTER_NEAREST))
    axes[row][1].imshow(gray,     cmap='gray', vmin=0, vmax=255)
    axes[row][2].imshow(enhanced, cmap='gray', vmin=0, vmax=255)
    axes[row][3].imshow(blurred,  cmap='gray', vmin=0, vmax=255)

    label = CIFAR10_CLASSES[y_all[idx]]
    axes[row][0].set_ylabel(label, fontsize=10, rotation=0, labelpad=55,
                             va='center', fontweight='bold')
    for ax in axes[row]:
        ax.axis('off')

fig.suptitle('Image Enhancement Comparison', fontsize=14, y=1.01)
plt.tight_layout()
plt.show()
print('✅ Enhancement demo complete!')

## 🤖 Step 5 — Train SVM and k-NN Models

> ⏱️ With `MAX_SAMPLES = 10 000` this takes roughly **2–5 minutes** in Colab.
> Increase to 60 000 for higher accuracy (SVM will take 30+ min).

In [ ]:
# ── Support Vector Machine ────────────────────────────────────────────────────
print('[SVM] Training… (RBF kernel, C=10, gamma=scale)')

svm = SVC(
    kernel='rbf',
    C=10,
    gamma='scale',
    random_state=RANDOM_STATE,
)
svm.fit(X_train, y_train)
print('✅ SVM training complete!')

In [ ]:
# ── k-Nearest Neighbours ──────────────────────────────────────────────────────
print('[k-NN] Training… (k=5, Euclidean distance)')

knn = KNeighborsClassifier(
    n_neighbors=5,
    metric='minkowski',
    n_jobs=-1,
)
knn.fit(X_train, y_train)
print('✅ k-NN training complete!')

## 📊 Step 6 — Evaluate Models

In [ ]:
def evaluate_model(model, X_test, y_test, model_name):
    """Print accuracy + classification report, draw confusion matrix."""

    y_pred   = model.predict(X_test)
    accuracy = accuracy_score(y_test, y_pred)

    print(f'\n{"="*54}')
    print(f'  {model_name}  —  Accuracy: {accuracy * 100:.2f}%')
    print(f'{"="*54}')
    print(classification_report(y_test, y_pred,
                                target_names=CIFAR10_CLASSES, digits=3))

    # ── Confusion matrix ──────────────────────────────────────────────────────
    cm = confusion_matrix(y_test, y_pred)

    fig, ax = plt.subplots(figsize=(11, 9))
    im = ax.imshow(cm, interpolation='nearest', cmap='Blues')
    fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)

    thresh = cm.max() / 2.0
    for i in range(cm.shape[0]):
        for j in range(cm.shape[1]):
            ax.text(j, i, str(cm[i, j]),
                    ha='center', va='center', fontsize=8,
                    color='white' if cm[i, j] > thresh else 'black')

    ticks = np.arange(len(CIFAR10_CLASSES))
    ax.set_xticks(ticks); ax.set_yticks(ticks)
    ax.set_xticklabels(CIFAR10_CLASSES, rotation=45, ha='right', fontsize=9)
    ax.set_yticklabels(CIFAR10_CLASSES, fontsize=9)
    ax.set_xlabel('Predicted Label', fontsize=11)
    ax.set_ylabel('True Label',      fontsize=11)
    ax.set_title(f'{model_name} — Confusion Matrix  (Accuracy: {accuracy*100:.2f}%)',
                 fontsize=13, pad=14)
    plt.tight_layout()
    plt.show()

    return accuracy, y_pred

In [ ]:
svm_acc, svm_preds = evaluate_model(svm, X_test, y_test, 'SVM')

In [ ]:
knn_acc, knn_preds = evaluate_model(knn, X_test, y_test, 'KNN')

In [ ]:
# ── Summary bar chart ─────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(5, 3.5))
bars = ax.bar(['SVM', 'k-NN'], [svm_acc * 100, knn_acc * 100],
              color=['#4C72B0', '#DD8452'], width=0.5, edgecolor='white')
for bar, val in zip(bars, [svm_acc, knn_acc]):
    ax.text(bar.get_x() + bar.get_width() / 2,
            bar.get_height() + 0.5,
            f'{val*100:.2f}%', ha='center', va='bottom', fontsize=12, fontweight='bold')
ax.set_ylim(0, 100)
ax.set_ylabel('Accuracy (%)', fontsize=11)
ax.set_title('Model Accuracy Comparison', fontsize=13)
ax.spines[['top', 'right']].set_visible(False)
plt.tight_layout()
plt.show()

print(f'\n  SVM  accuracy : {svm_acc*100:.2f}%')
print(f'  k-NN accuracy : {knn_acc*100:.2f}%')

## 🔍 Step 7 — Visualise Predictions

**Green title** = correct prediction &nbsp;|&nbsp; **Red title** = wrong prediction

In [ ]:
def show_predictions(y_pred, model_name, n=16):
    """4×4 grid of test images with predicted vs actual labels."""

    rng     = np.random.default_rng(seed=0)
    indices = rng.choice(len(X_test_raw_subset), size=n, replace=False)

    cols = int(np.sqrt(n))
    rows = n // cols
    fig, axes = plt.subplots(rows, cols, figsize=(cols * 2.8, rows * 3))
    fig.suptitle(
        f'{model_name} Predictions  —  '
        'green = correct  |  red = wrong',
        fontsize=13
    )

    for plot_idx, img_idx in enumerate(indices):
        ax = axes[plot_idx // cols][plot_idx % cols]

        actual    = CIFAR10_CLASSES[y_test[img_idx]]
        predicted = CIFAR10_CLASSES[y_pred[img_idx]]
        correct   = actual == predicted

        ax.imshow(X_test_raw_subset[img_idx])
        ax.set_title(
            f'Pred : {predicted}\nTrue : {actual}',
            color='green' if correct else 'red',
            fontsize=8.5
        )
        ax.axis('off')

    plt.tight_layout(rect=[0, 0, 1, 0.96])
    plt.show()

In [ ]:
show_predictions(svm_preds, 'SVM')

In [ ]:
show_predictions(knn_preds, 'KNN')

## 🏁 Done!

### Results Summary

| Model | Typical Accuracy (10k samples) |
|-------|--------------------------------|
| SVM (RBF) | ~40–45% |
| k-NN (k=5) | ~35–38% |

### Why not higher accuracy?
CIFAR-10 is genuinely hard. Classical ML on raw **grayscale pixels** tops out around 45%.  
Deep CNNs reach 90%+, but that's a different project!

### 💡 Try these tweaks right here in Colab

```python
MAX_SAMPLES = 30_000   # more data → better accuracy, slower training
IMG_SIZE    = 64       # larger features → richer info (also slower)
```

Or swap SVM for a Random Forest — just change one line in Step 5:
```python
from sklearn.ensemble import RandomForestClassifier
rf = RandomForestClassifier(n_estimators=200, n_jobs=-1, random_state=42)
rf.fit(X_train, y_train)
evaluate_model(rf, X_test, y_test, 'Random Forest')
```